In [ ]:
import random
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    cross_val_score,
    learning_curve,
    train_test_split
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from imblearn.pipeline import Pipeline 
from imblearn.over_sampling import SMOTE


# ============================================
# RANDOM SEED
# ============================================

random.seed(42)
np.random.seed(42)

print("Libraries loaded successfully.")

In [ ]:
# ============================================
# FILE CONFIGURATION
# ============================================

TRAIN_FILE = "datasets/subsidy_dataset.xlsx"

VALIDATION_FILE = "datasets/subsidy_validation_datasets.xlsx"

TARGET = "Effectiveness Label"

MODEL_FILE = "random_forest_subsidy.pkl"

print("Configuration loaded.")
print("Training file:", TRAIN_FILE)
print("Validation file:", VALIDATION_FILE)
print("Target:", TARGET)



In [ ]:
## Cell 3 — Load Training and Validation Datasets

# ============================================
# LOAD DATASETS
# ============================================

train_df = pd.read_excel(TRAIN_FILE)

validation_df = pd.read_excel(VALIDATION_FILE)


# ============================================
# DISPLAY TRAINING DATASET
# ============================================

print("\n========================================")
print("TRAINING DATASET")
print("========================================")

print("Shape:", train_df.shape)

display(train_df.head())


# ============================================
# DISPLAY VALIDATION DATASET
# ============================================

print("\n========================================")
print("EXTERNAL VALIDATION DATASET")
print("========================================")

print("Shape:", validation_df.shape)

display(validation_df.head())

In [ ]:
## Cell 4 — Dataset Information

# ============================================
# TRAINING DATASET INFORMATION
# ============================================

print("\n========================================")
print("TRAINING DATASET INFORMATION")
print("========================================")

train_df.info()


# ============================================
# VALIDATION DATASET INFORMATION
# ============================================

print("\n========================================")
print("VALIDATION DATASET INFORMATION")
print("========================================")

validation_df.info()

In [ ]:
## Cell 5 — Check Missing Values

# ============================================
# CHECK MISSING VALUES
# ============================================

print("\n========================================")
print("TRAINING MISSING VALUES")
print("========================================")

display(train_df.isnull().sum())


print("\n========================================")
print("VALIDATION MISSING VALUES")
print("========================================")

display(validation_df.isnull().sum())

In [ ]:
## Cell 6 — Check Target Distribution

# ============================================
# TARGET DISTRIBUTION
# ============================================

print("\n========================================")
print("TARGET DISTRIBUTION")
print("========================================")

target_counts = train_df[TARGET].value_counts()

display(target_counts)


print("\n========================================")
print("TARGET DISTRIBUTION (%)")
print("========================================")

target_percentage = (
    train_df[TARGET]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

display(target_percentage)

In [ ]:
## Cell 7 — Target Distribution Visualization
# ============================================
# TARGET DISTRIBUTION CHART
# ============================================

plt.figure(figsize=(8, 5))

train_df[TARGET].value_counts().plot(
    kind="bar"
)

plt.title("Effectiveness Label Distribution")
plt.xlabel("Effectiveness Label")
plt.ylabel("Number of Records")

plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()

plt.savefig(
    "target_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

plt.close()

In [ ]:
## Cell 8 — Define Features and Target
# ============================================
# FEATURES AND TARGET
# ============================================

X = train_df.drop(
    columns=[TARGET]
)

y = train_df[TARGET]


# ============================================
# EXTERNAL VALIDATION DATA
# ============================================

X_validation = validation_df.drop(
    columns=[TARGET]
)

y_validation = validation_df[TARGET]


print("========================================")
print("DATASET SHAPES")
print("========================================")

print("X:", X.shape)
print("y:", y.shape)

print("X_validation:", X_validation.shape)
print("y_validation:", y_validation.shape)

In [ ]:
## Cell 9 — Define Feature Types
# ============================================
# CATEGORICAL FEATURES
# ============================================

categorical_cols = [
    "Subsidy Type",
    "Pest",
    "Calamity"
]


# ============================================
# NUMERICAL FEATURES
# ============================================

numerical_cols = [
    "Farm Size (ha)",
    "Crop Yield Before",
    "Crop Yield After",
    "Income Before",
    "Income After",
    "Feedback Score"
]


print("========================================")
print("CATEGORICAL FEATURES")
print("========================================")

for column in categorical_cols:
    print("-", column)


print("\n========================================")
print("NUMERICAL FEATURES")
print("========================================")

for column in numerical_cols:
    print("-", column)

In [ ]:
## Cell 10 — Verify Required Columns

# ============================================
# VERIFY REQUIRED COLUMNS
# ============================================

required_columns = (
    categorical_cols +
    numerical_cols +
    [TARGET]
)

missing_training_columns = [
    column
    for column in required_columns
    if column not in train_df.columns
]

missing_validation_columns = [
    column
    for column in required_columns
    if column not in validation_df.columns
]


if missing_training_columns:
    print("Missing columns in training dataset:")
    print(missing_training_columns)

    raise ValueError(
        "Training dataset is missing required columns."
    )


if missing_validation_columns:
    print("Missing columns in validation dataset:")
    print(missing_validation_columns)

    raise ValueError(
        "Validation dataset is missing required columns."
    )


print("All required columns are present.")

In [ ]:
## Cell 11 — Create Preprocessor

# ============================================
# PREPROCESSING
# ============================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_cols
        ),

        (
            "num",
            "passthrough",
            numerical_cols
        )
    ]
)


print("Preprocessor created successfully.")

In [ ]:
## Cell 12 — 80/20 Train-Test Split

# ============================================
# TRAIN / TEST SPLIT
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)


print("========================================")
print("TRAIN / TEST SPLIT")
print("========================================")

print("Original dataset:", X.shape)

print(
    "Training dataset:",
    X_train.shape
)

print(
    "Testing dataset:",
    X_test.shape
)


print("\nTraining target distribution:")
display(
    y_train.value_counts()
)


print("\nTesting target distribution:")
display(
    y_test.value_counts()
)

In [ ]:
## Cell 13 — Create Random Forest Pipeline

# ============================================
# RANDOM FOREST PIPELINE
# ============================================

pipeline = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),

    (
        "classifier",
        RandomForestClassifier(
            random_state=42,
            n_jobs=-1
        )
    )
])


print("Random Forest pipeline created.")

In [ ]:
## Cell 14 — Hyperparameter Search Space
# ============================================
# HYPERPARAMETER SEARCH SPACE
# ============================================

params = {

    "classifier__n_estimators": [
        100,
        200,
        300,
        500
    ],

    "classifier__max_depth": [
        None,
        10,
        15,
        20,
        30
    ],

    "classifier__min_samples_split": [
        2,
        5,
        10
    ],

    "classifier__min_samples_leaf": [
        1,
        2,
        4
    ],

    "classifier__max_features": [
        "sqrt",
        "log2"
    ],

    "classifier__bootstrap": [
        True,
        False
    ]
}


print("Hyperparameter search space created.")

In [ ]:
## Cell 15 — RandomizedSearchCV

# ============================================
# RANDOMIZED SEARCH
# ============================================

search = RandomizedSearchCV(
    estimator=pipeline,

    param_distributions=params,

    n_iter=40,

    cv=5,

    scoring="accuracy",

    random_state=42,

    n_jobs=-1,

    verbose=1,

    return_train_score=True
)


print("========================================")
print("STARTING RANDOMIZED SEARCH")
print("========================================")

search.fit(
    X_train,
    y_train
)


# ============================================
# BEST MODEL
# ============================================

model = search.best_estimator_


print("\n========================================")
print("BEST PARAMETERS")
print("========================================")

print(search.best_params_)


print("\nBest Cross-Validation Score:")
print(
    f"{search.best_score_ * 100:.2f}%"
)

In [ ]:
## Cell 16 — Training and Testing Accuracy

# ============================================
# MODEL ACCURACY
# ============================================

train_acc = model.score(
    X_train,
    y_train
)

test_acc = model.score(
    X_test,
    y_test
)


print("\n========================================")
print("MODEL ACCURACY")
print("========================================")

print(
    f"Training Accuracy : "
    f"{train_acc * 100:.2f}%"
)

print(
    f"Testing Accuracy  : "
    f"{test_acc * 100:.2f}%"
)

In [ ]:
## Cell 17 — Generate Test Predictions

# ============================================
# TEST PREDICTIONS
# ============================================

y_pred = model.predict(
    X_test
)


print("Predictions generated.")

print("\nFirst 20 predictions:")

print(y_pred[:20])

In [ ]:
## Cell 18 — Classification Report

# ============================================
# CLASSIFICATION REPORT
# ============================================

print("\n========================================")
print("CLASSIFICATION REPORT")
print("========================================")

test_report = classification_report(
    y_test,
    y_pred,
    digits=4
)

print(test_report)

In [ ]:
## Cell 19 — Test Confusion Matrix

# ============================================
# TEST CONFUSION MATRIX
# ============================================

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=model.classes_
)


print("\n========================================")
print("TEST CONFUSION MATRIX")
print("========================================")

print(cm)


plt.figure(figsize=(8, 6))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=model.classes_
)

disp.plot(
    cmap="Blues",
    values_format="d"
)

plt.title(
    "Random Forest - Test Confusion Matrix"
)

plt.tight_layout()

plt.savefig(
    "confusion_matrix_test.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

plt.close()

In [ ]:
## Cell 20 — 5-Fold Cross Validation
# ============================================
# 5-FOLD CROSS VALIDATION
# ============================================

cv_scores = cross_val_score(
    model,
    X,
    y,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)


cv_mean = cv_scores.mean()
cv_std = cv_scores.std()


print("\n========================================")
print("5-FOLD CROSS VALIDATION")
print("========================================")


for i, score in enumerate(
    cv_scores,
    start=1
):
    print(
        f"Fold {i}: "
        f"{score * 100:.2f}%"
    )


print("\nMean CV Accuracy:")
print(
    f"{cv_mean * 100:.2f}%"
)


print("\nCV Standard Deviation:")
print(
    f"{cv_std * 100:.2f}%"
)

In [ ]:
## Cell 21 — Learning Curve
# ============================================
# LEARNING CURVE
# ============================================

train_sizes, train_scores, valid_scores = learning_curve(

    model,

    X,

    y,

    cv=5,

    train_sizes=np.linspace(
        0.1,
        1.0,
        10
    ),

    scoring="accuracy",

    n_jobs=-1
)


train_mean = train_scores.mean(
    axis=1
)

train_std = train_scores.std(
    axis=1
)

valid_mean = valid_scores.mean(
    axis=1
)

valid_std = valid_scores.std(
    axis=1
)


# ============================================
# PLOT
# ============================================

plt.figure(figsize=(9, 6))


plt.plot(
    train_sizes,
    train_mean,
    marker="o",
    label="Training Accuracy"
)


plt.plot(
    train_sizes,
    valid_mean,
    marker="o",
    label="Validation Accuracy"
)


plt.fill_between(
    train_sizes,
    train_mean - train_std,
    train_mean + train_std,
    alpha=0.15
)


plt.fill_between(
    train_sizes,
    valid_mean - valid_std,
    valid_mean + valid_std,
    alpha=0.15
)


plt.xlabel(
    "Training Samples"
)

plt.ylabel(
    "Accuracy"
)

plt.title(
    "Random Forest Learning Curve"
)

plt.grid(
    True,
    alpha=0.3
)

plt.legend()

plt.tight_layout()


plt.savefig(
    "learning_curve.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()

plt.close()

In [ ]:
## Cell 22 — External Validation
# ============================================
# EXTERNAL VALIDATION
# ============================================

val_pred = model.predict(
    X_validation
)


val_acc = accuracy_score(
    y_validation,
    val_pred
)


print("\n========================================")
print("EXTERNAL VALIDATION")
print("========================================")

print(
    f"External Validation Accuracy: "
    f"{val_acc * 100:.2f}%"
)


print("\nClassification Report:")

validation_report = classification_report(
    y_validation,
    val_pred,
    digits=4
)

print(validation_report)

In [ ]:
## Cell 23 — External Validation Confusion Matrix
# ============================================
# VALIDATION CONFUSION MATRIX
# ============================================

val_cm = confusion_matrix(
    y_validation,
    val_pred,
    labels=model.classes_
)


print("\n========================================")
print("EXTERNAL VALIDATION CONFUSION MATRIX")
print("========================================")

print(val_cm)


plt.figure(figsize=(8, 6))

disp = ConfusionMatrixDisplay(
    confusion_matrix=val_cm,
    display_labels=model.classes_
)

disp.plot(
    cmap="Blues",
    values_format="d"
)

plt.title(
    "Random Forest - External Validation Confusion Matrix"
)

plt.tight_layout()


plt.savefig(
    "confusion_matrix_validation.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()

plt.close()

In [ ]:
## Cell 24 — Feature Names
# ============================================
# FEATURE IMPORTANCE
# ============================================

feature_names = (
    model
    .named_steps["preprocessor"]
    .get_feature_names_out()
)


feature_importance = (
    model
    .named_steps["classifier"]
    .feature_importances_
)


print("\n========================================")
print("FEATURE IMPORTANCE")
print("========================================")

print(
    "Number of Features:",
    len(feature_names)
)

print(
    "Number of Importances:",
    len(feature_importance)
)

In [ ]:
## Cell 25 — Feature Importance Table
# ============================================
# FEATURE IMPORTANCE DATAFRAME
# ============================================

fi = pd.DataFrame({

    "Feature":
        feature_names,

    "Importance":
        feature_importance
})


fi = fi.sort_values(
    by="Importance",
    ascending=False
)


print("\n========================================")
print("TOP 20 MOST IMPORTANT FEATURES")
print("========================================")

display(
    fi.head(20)
)

In [ ]:
## Cell 26 — Feature Importance Chart
# ============================================
# FEATURE IMPORTANCE CHART
# ============================================

top_features = (
    fi
    .head(15)
    .sort_values(
        by="Importance"
    )
)


plt.figure(figsize=(10, 7))


plt.barh(
    top_features["Feature"],
    top_features["Importance"]
)


plt.xlabel(
    "Importance"
)

plt.ylabel(
    "Feature"
)

plt.title(
    "Top 15 Random Forest Feature Importances"
)


plt.tight_layout()


plt.savefig(
    "feature_importance.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()

plt.close()

In [ ]:
## Cell 27 — Export Feature Importance
# ============================================
# EXPORT FEATURE IMPORTANCE
# ============================================

FEATURE_FILE = "feature_importance.xlsx"


fi.to_excel(
    FEATURE_FILE,
    index=False
)


print(
    f"Feature importance saved to: "
    f"{FEATURE_FILE}"
)

In [ ]:
## Cell 28 — Calculate Detailed Metrics
# ============================================
# DETAILED METRICS
# ============================================

test_report_dict = classification_report(
    y_test,
    y_pred,
    output_dict=True
)


validation_report_dict = classification_report(
    y_validation,
    val_pred,
    output_dict=True
)


accuracy_metrics = pd.DataFrame({

    "Metric": [
        "Training Accuracy",
        "Testing Accuracy",
        "RandomizedSearchCV Best Score",
        "5-Fold CV Mean",
        "5-Fold CV Standard Deviation",
        "External Validation Accuracy"
    ],

    "Value": [
        train_acc,
        test_acc,
        search.best_score_,
        cv_mean,
        cv_std,
        val_acc
    ],

    "Percentage": [
        train_acc * 100,
        test_acc * 100,
        search.best_score_ * 100,
        cv_mean * 100,
        cv_std * 100,
        val_acc * 100
    ]
})


display(
    accuracy_metrics
)

In [ ]:
## Cell 29 — Export Metrics
# ============================================
# EXPORT METRICS
# ============================================

METRICS_FILE = "training_metrics.xlsx"


accuracy_metrics.to_excel(
    METRICS_FILE,
    index=False
)


print(
    f"Metrics saved to: "
    f"{METRICS_FILE}"
)

In [ ]:
## Cell 30 — Export Classification Reports
# ============================================
# EXPORT CLASSIFICATION REPORTS
# ============================================

test_report_df = pd.DataFrame(
    test_report_dict
).transpose()


validation_report_df = pd.DataFrame(
    validation_report_dict
).transpose()


with pd.ExcelWriter(
    "classification_reports.xlsx"
) as writer:

    test_report_df.to_excel(
        writer,
        sheet_name="Test"
    )

    validation_report_df.to_excel(
        writer,
        sheet_name="Validation"
    )


print(
    "classification_reports.xlsx created."
)

In [ ]:
## Cell 31 — Save the Trained Model
# ============================================
# SAVE MODEL
# ============================================

joblib.dump(
    model,
    MODEL_FILE
)


print("\n========================================")
print("MODEL SAVED")
print("========================================")

print(
    f"Model file: {MODEL_FILE}"
)

In [ ]:
## Cell 32 — Verify Saved Model
# ============================================
# LOAD AND VERIFY MODEL
# ============================================

loaded_model = joblib.load(
    MODEL_FILE
)


# Test prediction using loaded model
verification_prediction = loaded_model.predict(
    X_test.iloc[:5]
)


print("Model loaded successfully.")

print("\nSample predictions:")

print(
    verification_prediction
)

In [ ]:
## Cell 33 — Final Training Summary
# ============================================
# FINAL TRAINING SUMMARY
# ============================================

print("\n")
print("======================================================")
print("              AGRISUBSIDY MODEL TRAINING")
print("======================================================")

print(
    f"Training Dataset Size      : "
    f"{len(train_df):,}"
)

print(
    f"Training Set Size         : "
    f"{len(X_train):,}"
)

print(
    f"Testing Set Size          : "
    f"{len(X_test):,}"
)

print(
    f"External Validation Size  : "
    f"{len(X_validation):,}"
)

print("------------------------------------------------------")

print(
    f"Training Accuracy         : "
    f"{train_acc * 100:.2f}%"
)

print(
    f"Testing Accuracy          : "
    f"{test_acc * 100:.2f}%"
)

print(
    f"Random Search Best CV     : "
    f"{search.best_score_ * 100:.2f}%"
)

print(
    f"5-Fold CV Mean            : "
    f"{cv_mean * 100:.2f}%"
)

print(
    f"5-Fold CV Std             : "
    f"{cv_std * 100:.2f}%"
)

print(
    f"External Validation       : "
    f"{val_acc * 100:.2f}%"
)

print("------------------------------------------------------")

print("BEST PARAMETERS")

for parameter, value in search.best_params_.items():

    print(
        f"{parameter}: {value}"
    )

print("------------------------------------------------------")

print("GENERATED FILES")

print("- random_forest_subsidy.pkl")
print("- target_distribution.png")
print("- confusion_matrix_test.png")
print("- confusion_matrix_validation.png")
print("- learning_curve.png")
print("- feature_importance.png")
print("- feature_importance.xlsx")
print("- training_metrics.xlsx")
print("- classification_reports.xlsx")

print("======================================================")
print("             TRAINING COMPLETE")
print("======================================================")

In [ ]:
## Cell 34 — Optional: Test a New Prediction
# ============================================
# TEST NEW PREDICTION
# ============================================

new_farmer = pd.DataFrame({

    "Subsidy Type": [
        "Seeds"
    ],

    "Farm Size (ha)": [
        2.5
    ],

    "Crop Yield Before": [
        2500
    ],

    "Crop Yield After": [
        3500
    ],

    "Income Before": [
        50000
    ],

    "Income After": [
        75000
    ],

    "Feedback Score": [
        4.5
    ],

    "Pest": [
        "None"
    ],

    "Calamity": [
        "None"
    ]
})


prediction = loaded_model.predict(
    new_farmer
)


prediction_probability = (
    loaded_model.predict_proba(
        new_farmer
    )
)


print("\n========================================")
print("NEW FARMER PREDICTION")
print("========================================")

print(
    "Predicted Effectiveness:",
    prediction[0]
)


print("\nPrediction Probabilities:")

for class_name, probability in zip(
    loaded_model.classes_,
    prediction_probability[0]
):

    print(
        f"{class_name}: "
        f"{probability * 100:.2f}%"
    )